In [0]:
folder_path = '/Volumes/ai_capabilities/document_intelligence/invoices/load_invoice_1/test_invoices/test_invoices/'

In [0]:
from pyspark.sql import functions as F
new_df = (spark.read.format("binaryFile").load(f"{folder_path}img*.jpg")
    .select("path", F.expr("ai_parse_document(content, map('version','2.0')) AS parsed")))
new_df.write.mode("append").saveAsTable("ai_capabilities.document_intelligence.bronze_invoices_parsed")

In [0]:
from pyspark.sql import functions as F
ext_df = spark.read.table("ai_capabilities.document_intelligence.bronze_invoices_parsed").dropDuplicates(["path"])
extracted_df = ext_df.select("path", F.expr("""
                             ai_extract(
                             parsed,
                             '{
                                 "invoice_number": {"type":"string","description":"Invoice number of the seller invoice"},
      "invoice_date":   {"type":"string","description":"Invoice date, YYYY-MM-DD"},
      "vendor_vat_id":  {"type":"string","description":"VAT ID of the seller (issuer)"},
      "customer_vat_id":{"type":"string","description":"VAT ID of the customer (recipient)"},
      "total_amount":   {"type":"number","description":"Final total amount payable in EUR"},
      "line_items": {"type":"array","items":{"type":"object","properties":{
         "pos":{"type":"string"},"art_no":{"type":"string"},"description":{"type":"string"},
         "qty":{"type":"number"},"unit":{"type":"string"},
         "unit_price":{"type":"number"},"amount":{"type":"number"}}}}
                             }',
    map('version','2.1','mode','precision','enableConfidenceScores','true',
        'instructions','Vendor invoices. Ignore stamps and handwritten notes. Do not output the Carried forward or Carried over rows as line items.')
  ) AS extracted
"""))
display(extracted_df)

In [0]:
bronze_t = "ai_capabilities.document_intelligence.bronze_invoices_parsed"
have = spark.table(bronze_t).select("path")
imgs = spark.read.format("binaryFile").load(f"{folder_path}img*.jpg").join(have, "path", "left_anti")
imgs.select("path", F.expr("ai_parse_document(content, map('version','2.0')) AS parsed")).write.mode("append").saveAsTable(bronze_t)

In [0]:
from pyspark.sql import functions as F
bronze_t = "ai_capabilities.document_intelligence.bronze_invoices_parsed"
dc = "ai_capabilities.document_intelligence.silver_doc_class"
bronze = spark.table(bronze_t).dropDuplicates(["path"])
new_cls = bronze.join(spark.table(dc).select("path"), "path", "left_anti") \
  .select("path", F.element_at(F.split("path","/"),-1).alias("file_name"),
          F.expr("""ai_classify(parsed, '["invoice", "credit_note", "delivery_note", "purchase_order"]', map('version','2.1','enableConfidenceScores','true','instructions','Business documents received by accounts payable.')) AS cls"""))
new_cls.write.mode("append").saveAsTable(dc)

In [0]:
silver_t = "ai_capabilities.document_intelligence.silver_invoices_extracted"
# Route from the classification results themselves (invoice -> extract),
# so newly classified docs flow through without depending on an external routing table
route = (spark.table("ai_capabilities.document_intelligence.silver_doc_class")
           .select("path", F.expr("cls:response[0]:value::string").alias("doc_type"))
           .filter("doc_type = 'invoice'").select("path"))
todo = bronze.join(route, "path").join(spark.table(silver_t).select("path"), "path", "left_anti")
todo.select("path", F.expr(""" ai_extract(parsed, '{
    "invoice_number": {"type":"string","description":"Invoice number of the seller invoice"},
    "invoice_date":   {"type":"string","description":"Invoice date, YYYY-MM-DD"},
    "vendor_vat_id":  {"type":"string","description":"VAT ID of the seller (issuer)"},
    "customer_vat_id":{"type":"string","description":"VAT ID of the customer (recipient)"},
    "total_amount":   {"type":"number","description":"Final total amount payable in EUR"},
    "line_items": {"type":"array","items":{"type":"object","properties":{
       "pos":{"type":"string"},"art_no":{"type":"string"},"description":{"type":"string"},
       "qty":{"type":"number"},"unit":{"type":"string"},
       "unit_price":{"type":"number"},"amount":{"type":"number"}}}}
    }', map('version','2.1','mode','precision','enableConfidenceScores','true','instructions','Vendor invoices. Ignore stamps and handwritten notes. Do not output the Carried forward or Carried over rows as line items.')) AS extracted """)) \
    .write.mode("append").saveAsTable(silver_t)

In [0]:
s = (spark.table(silver_t)
       .withColumn("file_name", F.element_at(F.split("path","/"),-1))
       .withColumn("doc_key", F.regexp_replace("file_name", r"_p\d+\.(jpg|jpeg|png)$", "")))

header = (s.select("doc_key",
            F.expr("extracted:response:invoice_number:value::string").alias("invoice_number"),
            F.expr("extracted:response:invoice_date:value::string").alias("invoice_date"),
            F.expr("extracted:response:total_amount:value::decimal(18,2)").alias("total_amount"))
          .groupBy("doc_key").agg(F.max("invoice_number").alias("invoice_number"),
                                  F.max("invoice_date").alias("invoice_date"),
                                  F.max("total_amount").alias("total_amount")))

lines = (s.select("doc_key", F.explode(F.expr("try_cast(extracted:response:line_items AS ARRAY<VARIANT>)")).alias("li"))
          .select("doc_key", F.expr("li:pos:value::string").alias("pos"),
                  F.expr("li:amount:value::decimal(18,2)").alias("amount")))

check = (header.join(lines.groupBy("doc_key").agg(F.count("*").alias("n_lines"), F.sum("amount").alias("lines_sum")), "doc_key")
         .withColumn("diff", F.col("total_amount") - F.col("lines_sum")))

# Gold layer: persist the reconciled, validated invoice records (full recompute each run)
gold_t = "ai_capabilities.document_intelligence.gold_invoices"
check.write.mode("overwrite").saveAsTable(gold_t)
display(check)

# Pipeline overview

Incremental IDP pipeline: raw documents → parsed bronze → classified → extracted → reconciled gold.

* **Cell 1 — Source path setup.** Defines the Unity Catalog volume folder holding the test invoice documents (PDFs and JPG page images). Its `folder_path` variable feeds the file loads in Cells 2 and 4.
* **Cell 2 — Initial bronze parse.** Reads the `img*.jpg` page images as binary files and runs `ai_parse_document` on each, appending `path` + `parsed` (a variant) into the bronze table `bronze_invoices_parsed`. This seeds the bronze table that Cell 3 onward reads from.
* **Cell 3 — Schema-based extraction (preview).** Deduplicates the bronze table by `path` and runs `ai_extract` with the full invoice schema (invoice number, dates, VAT IDs, total, line items) to preview extracted fields. It establishes the exact schema and `ai_extract` options that Cell 6 later reuses for its incremental run.
* **Cell 4 — Incremental bronze ingestion.** Appends only new images not already in bronze (left-anti join on `path`), parsing them with `ai_parse_document`. Keeps the bronze table in sync without re-parsing, and its `bronze_t` variable/table feeds the classification step.
* **Cell 5 — Classify new documents.** Finds bronze docs not yet in `silver_doc_class` and classifies each parsed document with `ai_classify` into invoice / credit_note / delivery_note / purchase_order, appending results to `silver_doc_class`. The classification output drives downstream routing in Cell 6.
* **Cell 6 — Extract classified invoices.** Derives the route directly from Cell 5's classifications (`silver_doc_class`, doc_type = invoice), excludes docs already extracted in `silver_invoices_extracted`, and runs the same `ai_extract` call from Cell 3 on what remains. Its output table `silver_invoices_extracted` is exactly what Cell 7 consumes.
* **Cell 7 — Reconcile and write gold.** Reads the extracted silver table, groups multi-page documents by a `doc_key` (stripping `_p1/_p2` page suffixes), aggregates header fields and line items per document, reconciles `total_amount` against the line-item sum (`diff`), and writes the validated records to the gold table `gold_invoices` (full recompute each run).

**Overall flow:** Cell 1 (path) → Cells 2/4 (parse into bronze) → Cell 5 (classify) → Cell 6 (route + extract) → Cell 7 (validate + gold).

**Tables:** bronze `bronze_invoices_parsed` → silver `silver_doc_class` / `silver_invoices_extracted` → gold `gold_invoices`. The complete IDP chain runs end-to-end inside this notebook: images and PDFs alike are parsed, classified, extracted, reconciled (all docs currently pass with diff = 0.00), and landed in gold.